# WSI classification dataset (feature location and label) creation

## Example with TCGA-LUAD dataset (Normal_TS, Tumor_DX1)

### Step 1. Extract masks for the TCGA_LUAD dataset 

### Step 2. Extract features for the TCGA_LUAD dataset

##### Verifying the dimensions of features in .pt files (Optional)

In [2]:
import torch
import numpy as np
from pathlib import Path

def view_pt_file(filepath, show_data=False, max_rows=10):
    """
    View contents of a .pt file
    
    Args:
        filepath: Path to .pt file
        show_data: Whether to show actual data values (default: False)
        max_rows: Maximum rows to display if show_data=True (default: 10)
    """
    print("="*80)
    print(f"File: {filepath}")
    print("="*80)
    
    try:
        # Load the file
        data = torch.load(filepath, map_location='cpu', weights_only=False)
        
        # Determine data type
        if isinstance(data, torch.Tensor):
            print(f"\nType: PyTorch Tensor")
            print(f"Shape: {data.shape}")
            print(f"Dtype: {data.dtype}")
            print(f"Device: {data.device}")
            print(f"Requires grad: {data.requires_grad}")
            print(f"Memory: {data.element_size() * data.nelement() / (1024**2):.2f} MB")
            
            # Statistics
            if data.dtype in [torch.float32, torch.float64, torch.float16]:
                print(f"\nStatistics:")
                print(f"  Min: {data.min().item():.6f}")
                print(f"  Max: {data.max().item():.6f}")
                print(f"  Mean: {data.mean().item():.6f}")
                print(f"  Std: {data.std().item():.6f}")
            
            # Show data sample
            if show_data:
                print(f"\nData (first {max_rows} rows):")
                if len(data.shape) == 1:
                    print(data[:max_rows])
                elif len(data.shape) == 2:
                    print(data[:max_rows, :])
                else:
                    print(data.flatten()[:max_rows])
                    
        elif isinstance(data, dict):
            print(f"\nType: Dictionary")
            print(f"Keys: {list(data.keys())}")
            print(f"\nDetails:")
            for key, value in data.items():
                if isinstance(value, torch.Tensor):
                    print(f"  {key}: Tensor{tuple(value.shape)} ({value.dtype})")
                else:
                    print(f"  {key}: {type(value).__name__}")
                    
        elif isinstance(data, list):
            print(f"\nType: List")
            print(f"Length: {len(data)}")
            if data:
                print(f"First element type: {type(data[0]).__name__}")
                if isinstance(data[0], torch.Tensor):
                    print(f"First element shape: {data[0].shape}")
                    
        elif isinstance(data, np.ndarray):
            print(f"\nType: NumPy Array")
            print(f"Shape: {data.shape}")
            print(f"Dtype: {data.dtype}")
            print(f"Memory: {data.nbytes / (1024**2):.2f} MB")
            
        else:
            print(f"\nType: {type(data).__name__}")
            print(f"Value: {data}")
            
    except Exception as e:
        print(f"\nError loading file: {e}")
    
    print("="*80 + "\n")


def quick_view(filepath):
    """Quick view - just basic info"""
    try:
        data = torch.load(filepath, map_location='cpu', weights_only=False)
        if isinstance(data, torch.Tensor):
            print(f"{Path(filepath).name}: Tensor{tuple(data.shape)} {data.dtype} ({data.element_size() * data.nelement() / (1024**2):.2f} MB)")
        elif isinstance(data, dict):
            print(f"{Path(filepath).name}: Dict with keys {list(data.keys())}")
        else:
            print(f"{Path(filepath).name}: {type(data).__name__}")
    except Exception as e:
        print(f"{Path(filepath).name}: Error - {e}")


def batch_view(folder, pattern="*.pt", quick=True):
    """View multiple .pt files in a folder"""
    folder_path = Path(folder)
    files = sorted(list(folder_path.glob(pattern)))
    
    print(f"Found {len(files)} files matching '{pattern}' in {folder}\n")
    
    for file in files:
        if quick:
            quick_view(file)
        else:
            view_pt_file(file)


# Example usage
if __name__ == "__main__":
    # Single file - detailed view
    path = "/data_64T_3/Raja/Extracted_Features/CLAM/TCGA_LUAD/Normal_TS/Extracted_Features/resnet18/TCGA-38-4630-11A-01-TS1.00ed18d6-acf8-4950-bb49-6a8dc011ef6e.pt"
    
    view_pt_file(path, show_data=True, max_rows=5) 
    
    # Quick view
    quick_view(path)  

File: /data_64T_3/Raja/Extracted_Features/CLAM/TCGA_LUAD/Normal_TS/Extracted_Features/resnet18/TCGA-38-4630-11A-01-TS1.00ed18d6-acf8-4950-bb49-6a8dc011ef6e.pt

Type: PyTorch Tensor
Shape: torch.Size([1311, 512])
Dtype: torch.float32
Device: cpu
Requires grad: False
Memory: 2.56 MB

Statistics:
  Min: 0.000000
  Max: 9.678200
  Mean: 0.635619
  Std: 0.718399

Data (first 5 rows):
tensor([[0.9696, 0.9552, 0.0300,  ..., 0.4777, 0.3152, 0.8445],
        [0.6505, 0.0303, 0.4084,  ..., 0.6925, 0.0658, 0.3068],
        [0.8483, 0.0000, 0.3972,  ..., 0.5921, 0.0000, 0.4081],
        [0.5628, 0.0693, 0.1629,  ..., 0.7038, 0.0306, 0.7780],
        [0.6776, 0.0042, 0.3615,  ..., 0.9191, 0.0822, 0.1765]])

TCGA-38-4630-11A-01-TS1.00ed18d6-acf8-4950-bb49-6a8dc011ef6e.pt: Tensor(1311, 512) torch.float32 (2.56 MB)


### Step 3: Prepare a csv file with slide_name and label for TCGA_LUAD

In [18]:
#!/usr/bin/env python3
"""Create slide CSV with custom Normal=0, Tumor=1 mapping for the dataset with an example TCGA_LUAD."""
import pandas as pd
from pathlib import Path
import sys

# Configuration
dataset = 'TCGA_LUAD'
input_path = f'/data_64T_2/Dataset/{dataset}/images'
extensions = ['*.svs', '*.tif', '*.tiff']
tissues = ['Normal_TS', 'Tumor_DX1']

output_path = f'/data_64T_2/Dataset/{dataset}/labels/{dataset}_binary_class_label.csv'

# Exclusion list - add WSI names (without extension) to exclude.
# Because of no MPP or no mask, no features would have been extracted. so include those slide names as below.
exclude_list = [  
    # 'slide_name_1',
    # 'slide_name_2',
    # Add more slide names here as needed
]

# Custom label mapping
label_map = {tissues[0] : 0, tissues[1] : 1}

# Generate data
data = []
for tissue in tissues:
    tissue_folder = Path(input_path) / tissue
    if not tissue_folder.exists():
        print(f"⚠️  Warning: {tissue_folder} does not exist, skipping...")
        continue
    
    # Check if it's a directory structure or files directly in tissue folder
    svs_files = [f for ext in extensions for f in tissue_folder.glob(ext)]
    
    # print(f"tissue_folder: {tissue_folder}")
    # print(f"len(svs_files) : {len(svs_files)}")

    if svs_files:
        # Files are directly in the tissue folder
        label = label_map[tissue]
        for image in sorted(svs_files):
            if image.stem not in exclude_list:
                data.append({'slide_name': image.stem, 'label': label})

# # Save CSV
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame(data)
df.to_csv(output_path, index=False)

print(f"✓ Created {output_path}")
print(f"  Total slides: {len(df)}")
print(f"  Excluded slides: {len(exclude_list)}")
print(f"  Normal (0): {(df['label']==0).sum()}")
print(f"  Tumor (1): {(df['label']==1).sum()}")

✓ Created /data_64T_2/Dataset/TCGA_KIRC/labels/TCGA_KIRC_binary_class_label.csv
  Total slides: 927
  Excluded slides: 20
  Normal (0): 416
  Tumor (1): 511


### Step 4: Prepare k-fold common train-test splits to run with different MIL model

### Step 5: Replacing slide_name with its feature path from different encdoer

In [9]:
#!/usr/bin/env python3
"""Add feature paths to WSI names in CSV files."""
import pandas as pd
from pathlib import Path
import sys

base_path = "/data_64T_3/Shared/Extracted_Features"
dataset = "TCGA_LUAD"    # TCGA_LUAD
pre_processor = "TRIDENT"   # MUFASA, TRIDENT, CLAM, HISTOLAB 
tissue = ["Normal_TS", "Tumor_DX1"]
encoder = "resnet50_1024"   # 'resnet18', 'uni'

# Configuration
csv_dir = f'/data_64T_2/Dataset/{dataset}/labels/5-fold-common-split' # path that contains split inforamation. 
# Even if there is a single csv file, place it in a folder and provide its location

# feature location
feature_dirs = [f'{base_path}/{pre_processor}/{dataset}/{tissue[0]}/Extracted_features/{encoder}',
                f'{base_path}/{pre_processor}/{dataset}/{tissue[1]}/Extracted_features/{encoder}']

# output path to store
# output_path = f"/home/rajaj/Project/7.Multi_Phase_Tile_Extractor_Experiments/WSI_Classification/MIL_BASELINE/datasets/{dataset}/{dataset}_{pre_processor}_{encoder}_splits"
output_path = f"{base_path}/{pre_processor}/{dataset}/{dataset}_{pre_processor}_{encoder}_splits"
Path(output_path).mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------------------------------------
# Build feature lookup dictionary
print("Building feature file index...")
feature_map = {}
dirs_found = True # Flag to track if all feature directories exist

for feature_dir in feature_dirs:
    # Check if directory exists to avoid errors
    if Path(feature_dir).exists():
        for pt_file in Path(feature_dir).rglob('*.pt'):
            slide_name = pt_file.stem
            feature_map[slide_name] = str(pt_file)
    else:
        print(f"Warning: Directory not found: {feature_dir}")
        dirs_found = False # Set flag to False if any directory is missing

if not dirs_found:
    print("⚠️  Stopping: One or more feature directories are missing.")
    sys.exit(1) # Exit script if directories are bad

print(f"Found {len(feature_map)} feature files across {len(feature_dirs)} directories")

# Process all CSV files
for csv_file in Path(csv_dir).glob('*.csv'):
    df = pd.read_csv(csv_file)
    missing = []
    
    # Update all *_slide_path columns
    for col in df.columns:
        if col.endswith('_slide_path'):
            def get_path(x):
                if pd.notna(x):
                    # Ensure x is a string to match dictionary keys safely
                    x_str = str(x).strip() 
                    if x_str in feature_map:
                        return feature_map[x_str]
                    else:
                        missing.append(x_str)
                        return x  # Keep original if not found
                return x
            
            df[col] = df[col].apply(get_path)
    
    # --- CHANGED LOGIC HERE ---
    if missing:
        unique_missing = set(missing)
        print(f"⚠️  SKIPPING {csv_file.name}: Missing {len(unique_missing)} features: {unique_missing}")
        # Do NOT save the file, simply continue to the next one

    # Only reach here if missing list is empty
    output_file = f'{output_path}/{csv_file.stem}_with_path.csv'
    df.to_csv(output_file, index=False)
    
    print(f"✓ {csv_file.name} -> {Path(output_file).name}")

Building feature file index...
Found 793 feature files across 1 directories
✓ Total_5-fold_Stanford_5fold.csv -> Total_5-fold_Stanford_5fold_with_path.csv
✓ Total_5-fold_Stanford_4fold.csv -> Total_5-fold_Stanford_4fold_with_path.csv
✓ Total_5-fold_Stanford_2fold.csv -> Total_5-fold_Stanford_2fold_with_path.csv
✓ Total_5-fold_Stanford_1fold.csv -> Total_5-fold_Stanford_1fold_with_path.csv
✓ Total_5-fold_Stanford_3fold.csv -> Total_5-fold_Stanford_3fold_with_path.csv


In [38]:
#!/usr/bin/env python3
"""Add feature paths to WSI names in CSV files."""
import pandas as pd
from pathlib import Path
import sys

# base_path = "/data_64T_3/Shared/Extracted_Features"
dataset = "STANFORD_793"    # TCGA_LUAD
pre_processor = "TRIDENT"   # MUFASA, TRIDENT, CLAM, HISTOLAB 
# tissue = ["Normal_TS", "Tumor_DX1"]
encoder = "uni"   # 'resnet18', 'uni'

# Configuration
csv_dir = f'/data_64T_2/Dataset/STANFORD_793/labels/MSS_MSI/STANFORD_793_SPLITS' # path that contains split inforamation. 
# Even if there is a single csv file, place it in a folder and provide its location

# feature location
feature_dirs = [f'/data_64T_3/Shared/Extracted_Features/{pre_processor}/STANFORD_793/Tumor/Extracted_features/resnet50_1024']

# output path to store
# output_path = f"/home/rajaj/Project/7.Multi_Phase_Tile_Extractor_Experiments/WSI_Classification/MIL_BASELINE/datasets/{dataset}/{dataset}_{pre_processor}_{encoder}_splits"
output_path = f"/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/New_folder/{dataset}_{pre_processor}_{encoder}_splits"
Path(output_path).mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------------------------------------
# Build feature lookup dictionary
print("Building feature file index...")
feature_map = {}
dirs_found = True # Flag to track if all feature directories exist

for feature_dir in feature_dirs:
    # Check if directory exists to avoid errors
    if Path(feature_dir).exists():
        for pt_file in Path(feature_dir).rglob('*.pt'):
            slide_name = pt_file.stem
            feature_map[slide_name] = str(pt_file)
    else:
        print(f"Warning: Directory not found: {feature_dir}")
        dirs_found = False # Set flag to False if any directory is missing

if not dirs_found:
    print("⚠️  Stopping: One or more feature directories are missing.")
    sys.exit(1) # Exit script if directories are bad

print(f"Found {len(feature_map)} feature files across {len(feature_dirs)} directories")

# Process all CSV files
for csv_file in Path(csv_dir).glob('*.csv'):
    df = pd.read_csv(csv_file)
    missing = []
    
    # Update all *_slide_path columns
    for col in df.columns:
        if col.endswith('_slide_path'):
            def get_path(x):
                if pd.notna(x):
                    # Ensure x is a string to match dictionary keys safely
                    x_str = str(x).strip() 
                    if x_str in feature_map:
                        return feature_map[x_str]
                    else:
                        missing.append(x_str)
                        return x  # Keep original if not found
                return x
            
            df[col] = df[col].apply(get_path)
    
    # --- CHANGED LOGIC HERE ---
    if missing:
        unique_missing = set(missing)
        print(f"⚠️  SKIPPING {csv_file.name}: Missing {len(unique_missing)} features: {unique_missing}")
        # Do NOT save the file, simply continue to the next one

    # Only reach here if missing list is empty
    output_file = f'{output_path}/{csv_file.stem}_with_path.csv'
    df.to_csv(output_file, index=False)
    
    print(f"✓ {csv_file.name} -> {Path(output_file).name}")

Building feature file index...
Found 778 feature files across 1 directories
⚠️  SKIPPING Total_5-fold_Stanford_793_3fold.csv: Missing 14 features: {'P-2824', 'P-2853', 'P-3673', 'P-3657', 'P-3710', 'P-1208', 'P-2677a', 'P-2403', 'P-1408', 'P-3014', 'P-0515', 'P-3663', 'P-0511', 'P-1622'}
✓ Total_5-fold_Stanford_793_3fold.csv -> Total_5-fold_Stanford_793_3fold_with_path.csv
⚠️  SKIPPING Total_5-fold_Stanford_793_2fold.csv: Missing 14 features: {'P-2824', 'P-2853', 'P-3673', 'P-2677a', 'P-1208', 'P-3657', 'P-3710', 'P-2403', 'P-1408', 'P-3014', 'P-0515', 'P-3663', 'P-0511', 'P-1622'}
✓ Total_5-fold_Stanford_793_2fold.csv -> Total_5-fold_Stanford_793_2fold_with_path.csv
⚠️  SKIPPING Total_5-fold_Stanford_793_1fold.csv: Missing 14 features: {'P-2824', 'P-2853', 'P-3673', 'P-2677a', 'P-1208', 'P-3657', 'P-3710', 'P-2403', 'P-1408', 'P-3014', 'P-0515', 'P-3663', 'P-0511', 'P-1622'}
✓ Total_5-fold_Stanford_793_1fold.csv -> Total_5-fold_Stanford_793_1fold_with_path.csv
⚠️  SKIPPING Total_5-fol

### Step 6: Update the dataset location in MIL_BASELINE/configs/CLAM_MB.yaml file

In [ ]:
Update the parameters: DATASET_NAME, dataset_root_dir, log_root_dir carefully

### Step 7: or override these parameter while launching the MIL model

### Step 8: Run test on a set of WSI

In [ ]:
Input dataset: Test.csv
slide_path	label
/path/WSI1.pt	0
/path/WSI2.pt	1

In [ ]:
python test_mil.py \
--yaml_path configs/CLAM_MB_MIL.yaml \
--test_dataset_csv datasets/Inference.csv \
--model_weight_path logs/path/Best_EPOCH_57.pth \
--test_log_dir logs/Test_output

In [ ]:
Output folder strucutre 
logs/Test_output
    |-Test_CLAM_MB_MIL.yaml
    |-Test_dataset_TCGA_LUAD.csv
    |-Test_Log_CLAM_MB_MIL.txt

### Step 9: Inference on a set of WSI that does not label

In [ ]:
python test_mil.py \
--yaml_path configs/CLAM_MB_MIL.yaml \
--test_dataset_csv datasets/Inference.csv \
--model_weight_path logs/path/Best_EPOCH_57.pth \
--test_log_dir logs/Inference_output

In [ ]:
Output folder contains a .csv file with the following data

slide_path	  slide_name	prediction	         probabilities
/path/WSI1.pt    WSI1           0	    [0.9865760803222656, 0.013423897325992584]
/path/WSI2.pt	 WSI2           1	    [0.1328595131635666, 0.867140531539917]

### Step 10: Generating slide level visualizations

##### Draw Feature Map

##### Draw Attention Map